# langchain语言模型介绍
一个 AI 应用的核心就是它所依赖的大语言模型，LangChain作为一个“工具”，不提供任何 LLMs，而是依赖于第三方集成各种大模型。比如，将 OpenAI、Anthropic、Hugging Face 、LlaMA、阿里Qwen、ChatGLM等平台的模型无缝接入到你的应用。
LangChain 模型接口可参考官方文档：https://reference.langchain.com/python/langchain_core/language_models/

LangChain中将大语言模型分为以下几种，我们主要使用的是聊天模型：
![alt text](img/image1.png)

## ChatModel主要参数
![alt text](img/image2.png)

以上的标准参数，也只是适用于部分的大语言模型，有些参数在特定模型中可能是无效的，这些标准化参数仅对 LangChain 官方提供集成包的模型（如 langchain-openai、langchain-anthropic）生效，在langchain-community包中的第三方模型，则不需要遵守这些标准化参数的规则。

## Message组件
调用模型后返回了一条AI消息，在LangChain中，消息有几种不同的类型。所有消息都有 type 、 content 、 response_metadata 等属性。
下面是这几个属性的作用：
![alt text](img/image3.png)


## 安装Langchain包

In [ ]:
!pip install langchain
!pip install langchain-deepseek
!pip install langchain-ollama
!pip install langchain-community

### 创建秘钥环境变量

In [1]:
! touch .env

In [ ]:
# 根据自己的API密钥添加下面内容到.env文件中
DEEPSEEK_API_KEY=XXXX
QWEN_API_KEY=XXXX
OPENAI_API_KEY=XXX

读取变量到当前环境中

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
print(f"Deepseek API Key: {deepseek_api_key}")

Deepseek API Key: sk-65a790441b194d7387f467f8250d24ff


调用聊天模型

In [4]:
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek

# llms = ChatOpenAI()
llms_ds = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=deepseek_api_key,
)
print(llms_ds.invoke("什么是LangChain?"))


content='**LangChain** 是一个用于构建**大语言模型（LLM）驱动应用程序**的开源框架。它的核心目标是简化将 LLM（如 GPT-4、Llama 等）与外部数据源和工具连接起来的过程，从而创建功能强大、实用的应用。\n\n你可以把它想象成 LLM 世界的“粘合剂”或“工具箱”。\n\n### 核心解决的问题\n单纯使用大语言模型 API（如直接调用 ChatGPT）存在一些限制：\n1.  **知识截止性**：模型训练数据有截止日期，不知道最新信息。\n2.  **缺乏特定领域知识**：模型不了解非公开的、私有的数据（如公司内部文档）。\n3.  **“幻觉”问题**：模型可能会编造看似合理但错误的答案。\n4.  **无法执行具体操作**：模型本身不能查数据库、调用 API 或进行数学计算。\n\n**LangChain 就是为了解决这些问题而生的**，它提供了一套标准化的接口和组件，让开发者能轻松地：\n*   **为 LLM 接入最新、特定的数据**。\n*   **让 LLM 能够使用工具**。\n*   **构建多步骤、有状态的复杂推理链**。\n\n---\n\n### 核心概念与组件\n\nLangChain 的架构围绕以下几个关键模块构建：\n\n1.  **模型（Models）**\n    *   **LLMs**：各种大语言模型的抽象接口（如 OpenAI、Anthropic、Llama 等）。\n    *   **聊天模型**：专为对话优化的模型（如 GPT-4）。\n    *   **嵌入模型**：将文本转换为数值向量，用于搜索和比较。\n\n2.  **提示（Prompts）**\n    *   管理、优化和模板化输入给模型的指令（提示词）。支持动态提示，可以根据用户输入插入不同的上下文。\n\n3.  **链（Chains）**\n    *   **这是 LangChain 的灵魂**。它将多个组件“链”在一起，完成一个复杂的任务。\n    *   例如，一个链可以：`接收用户问题 -> 检索相关文档 -> 将文档和问题组合成提示 -> 发送给 LLM -> 解析输出`。\n    *   提供了现成的常用链（如 `RetrievalQA`），也支持自定义链。\n\n4.  **检索（Retrieval

In [3]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv(override=True)

client = OpenAI(
    # 如果没有配置环境变量，请用阿里云百炼API Key替换：api_key="sk-xxx"
    api_key=os.getenv("QWEN_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

messages = [{"role": "user", "content": "你是谁"}]
completion = client.chat.completions.create(
    model="qwen3.6-plus",  # 您可以按需更换为其它深度思考模型
    messages=messages,
    extra_body={"enable_thinking": True},
    stream=True
)
is_answering = False  # 是否进入回复阶段
print("\n" + "=" * 20 + "思考过程" + "=" * 20)
for chunk in completion:
    delta = chunk.choices[0].delta
    if hasattr(delta, "reasoning_content") and delta.reasoning_content is not None:
        if not is_answering:
            print(delta.reasoning_content, end="", flush=True)
    if hasattr(delta, "content") and delta.content:
        if not is_answering:
            print("\n" + "=" * 20 + "完整回复" + "=" * 20)
            is_answering = True
        print(delta.content, end="", flush=True)


====================思考过程====================
Here's a thinking process:

1.  **Analyze User Input:**
   - User asks: "你是谁" (Who are you?)
   - Language: Chinese
   - Intent: Identify the AI assistant

2.  **Identify Key Constraints/Requirements:**
   - I need to respond in Chinese.
   - I should state my identity clearly and concisely.
   - I am Qwen (通义千问), developed by Alibaba Group's Tongyi Lab.

3.  **Formulate Response (Internal Refinement):**
   - Direct answer: 我是 Qwen（通义千问），由阿里巴巴集团旗下通义实验室自主研发的大语言模型。
   - Keep it polite and helpful.
   - Ensure it matches the required identity.

   Draft: 我是 Qwen（通义千问），由阿里巴巴集团旗下通义实验室自主研发的大语言模型。有什么我可以帮你的吗？

4.  **Self-Correction/Verification:**
   - Check identity: Qwen / 通义千问, developed by Alibaba Group's Tongyi Lab. (Matches requirement)
   - Language: Chinese. (Matches user input)
   - Tone: Helpful, concise. (Good)
   - No extra fluff. (Good)

   The response is ready.✅
====================完整回复====================
我是 Qwen（通义千问），由阿里巴巴集团旗下通义实验

IndexError: list index out of range